# Airport Traffic EDA

European IFR flight movements (2019–2026). Source: EUROCONTROL.

## 1. Setup

In [ ]:
from pathlib import Path


def find_project_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "data").exists():
            return path
    raise RuntimeError("Could not find project root")


PROJECT_ROOT = find_project_root()
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)


def save_figure(fig, name: str) -> None:
    fig.savefig(FIGURE_DIR / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIGURE_DIR / f"{name}.pdf", bbox_inches="tight")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns


In [ ]:
BLUE   = "#2176AE"
ORANGE = "#E76F51"
TEAL   = "#2A9D8F"
SLATE  = "#264653"
RED    = "#E63946"
GRAY   = "#C8C8C8"

plt.rcParams.update({
    "figure.dpi":          130,
    "figure.facecolor":    "white",
    "axes.facecolor":      "white",
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "axes.grid":           False,
    "font.family":         "sans-serif",
    "axes.titlesize":      12,
    "axes.titleweight":    "bold",
    "axes.labelsize":      10,
    "xtick.labelsize":     9,
    "ytick.labelsize":     9,
    "legend.fontsize":     9,
    "legend.frameon":      False,
    "figure.titlesize":    13,
    "figure.titleweight":  "bold",
})

## 2. Load Clean Data

In [ ]:
df = pd.read_csv(CLEAN_DIR / "airport_clean.csv", parse_dates=["FLT_DATE"])
print(f"Shape: {df.shape}")
df.info()
df.sample(5)


## 3. Exploratory Data Analysis

### 3.1 Data Coverage Over Time

In [ ]:
cov = (
    df.groupby(df["FLT_DATE"].dt.to_period("M"))["APT_ICAO"]
    .nunique()
    .reset_index()
)
cov.columns = ["period", "n_airports"]
cov["date"] = cov["period"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(cov["date"], cov["n_airports"], alpha=0.12, color=BLUE)
ax.plot(cov["date"], cov["n_airports"], color=BLUE, lw=2)
ax.axvline(pd.Timestamp("2020-03-15"), color=RED, ls="--", lw=1.2, label="COVID")
ax.set_ylabel("Airports")
ax.set_title("Airports with available data per month")
ax.legend()
plt.tight_layout()
save_figure(fig, "airport_data_coverage")
plt.show()

### 3.2 Daily Traffic Timeline

In [ ]:
daily = df.groupby("FLT_DATE")["FLT_TOT_1"].sum().reset_index()
daily = daily.sort_values("FLT_DATE")
daily["rolling7"] = daily["FLT_TOT_1"].rolling(7, center=True).mean()

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(daily["FLT_DATE"], daily["FLT_TOT_1"] / 1e3,
        color=GRAY, lw=0.4, alpha=0.7, label="Daily")
ax.plot(daily["FLT_DATE"], daily["rolling7"] / 1e3,
        color=BLUE, lw=1.8, label="7-day rolling avg")

covid_start = pd.Timestamp("2020-03-01")
covid_end   = pd.Timestamp("2020-07-01")
covid_mid   = covid_start + (covid_end - covid_start) / 2

ax.axvspan(covid_start, covid_end, alpha=0.07, color=RED)
ax.text(covid_mid, 8, "COVID",
        fontsize=7, color=RED, fontweight="bold", ha="center")

peak_idx  = daily["FLT_TOT_1"].idxmax()
peak_date = daily.loc[peak_idx, "FLT_DATE"]
peak_val  = daily.loc[peak_idx, "FLT_TOT_1"]
ax.annotate(
    f"Peak: {peak_date.strftime('%d/%m/%Y')}\n{peak_val:,.0f} flights",
    xy=(peak_date, peak_val / 1e3),
    xytext=(peak_date - pd.Timedelta(days=250), peak_val / 1e3 + 6),
    fontsize=8,
    arrowprops=dict(arrowstyle="->", color="#555"),
    color="#555",
)

ax.set_ylabel("Total flights (thousands)")
ax.set_title("European daily air traffic (2019–2026)")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.tight_layout()
save_figure(fig, "airport_daily_traffic")
plt.show()

### 3.3 Seasonal Patterns by Year

In [ ]:
monthly_avg = df.groupby(["YEAR", "MONTH_NUM"])["FLT_TOT_1"].sum().reset_index()
days_in_month = (
    df.groupby(["YEAR", "MONTH_NUM"])["FLT_DATE"]
    .nunique()
    .reset_index(name="n_days")
)
monthly_avg = monthly_avg.merge(days_in_month)
monthly_avg["daily_avg"] = monthly_avg["FLT_TOT_1"] / monthly_avg["n_days"]

piv_s = monthly_avg.pivot(index="MONTH_NUM", columns="YEAR", values="daily_avg")

HIGHLIGHT  = {2019: BLUE, 2020: RED, 2024: TEAL}
MON_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
              "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, ax = plt.subplots(figsize=(12, 5))
for yr in piv_s.columns:
    if yr in HIGHLIGHT:
        ax.plot(piv_s.index, piv_s[yr] / 1e3, marker="o", ms=4,
                color=HIGHLIGHT[yr], lw=2.0, label=str(yr), zorder=3)
    else:
        ax.plot(piv_s.index, piv_s[yr] / 1e3, marker="o", ms=2,
                color=GRAY, lw=1.0, alpha=0.5, label="_nolegend_", zorder=1)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(MON_LABELS)
ax.set_ylabel("Avg daily flights (thousands)")
ax.set_title("Seasonal patterns: daily average by month")
ax.legend(title="Highlighted years")
plt.tight_layout()
save_figure(fig, "airport_seasonal_patterns")
plt.show()

### 3.4 Top 25 Airports by Total Volume

In [ ]:
top25 = (
    df.groupby(["APT_ICAO", "APT_NAME"])["FLT_TOT_1"]
    .sum()
    .reset_index()
    .sort_values("FLT_TOT_1", ascending=False)
    .head(25)
)

fig, ax = plt.subplots(figsize=(12, 7))
labels = [f"{row.APT_NAME} ({row.APT_ICAO})" for _, row in top25.iterrows()]
bars   = ax.barh(labels, top25["FLT_TOT_1"] / 1e6, color=SLATE, edgecolor="white")
ax.invert_yaxis()
ax.set_xlabel("Total flights 2019–2026 (millions)")
ax.set_title("Top 25 European airports by traffic volume")

for bar, val in zip(bars, top25["FLT_TOT_1"]):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
            f"{val / 1e6:.1f}M", va="center", fontsize=8, color="#444")

plt.tight_layout()
save_figure(fig, "airport_top25_volume")
plt.show()

### 3.5 Airport Volume Distribution: 2019 vs 2024

In [ ]:
apt_year = (
    df[df["YEAR"].isin([2019, 2024])]
    .groupby(["APT_ICAO", "APT_NAME", "YEAR"])["FLT_TOT_1"]
    .sum()
    .reset_index()
)
apt_piv = (
    apt_year
    .pivot(index=["APT_ICAO", "APT_NAME"], columns="YEAR", values="FLT_TOT_1")
    .dropna()
    .rename(columns={2019: "y2019", 2024: "y2024"})
)
apt_piv["log2019"] = np.log10(apt_piv["y2019"].clip(lower=1))
apt_piv["log2024"] = np.log10(apt_piv["y2024"].clip(lower=1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, col, color in [("2019", "log2019", BLUE), ("2024", "log2024", ORANGE)]:
    axes[0].hist(apt_piv[col], bins=30, alpha=0.55, color=color,
                 label=label, edgecolor="white")
axes[0].set_xlabel("log\u2081\u2080(Annual flights)")
axes[0].set_ylabel("Number of airports")
axes[0].set_title("Airport volume distribution: 2019 vs 2024")
axes[0].legend()

maxval = max(apt_piv["y2019"].max(), apt_piv["y2024"].max()) / 1e3 * 1.05
axes[1].scatter(apt_piv["y2019"] / 1e3, apt_piv["y2024"] / 1e3,
                s=30, alpha=0.5, color=SLATE, edgecolors="white", lw=0.3)
axes[1].plot([0, maxval], [0, maxval], color=RED, ls="--", lw=1.2, label="Parity")
axes[1].set_xlabel("Annual flights 2019 (thousands)")
axes[1].set_ylabel("Annual flights 2024 (thousands)")
axes[1].set_title("2019 vs 2024 traffic per airport")
axes[1].legend()

apt_piv_r = apt_piv.reset_index()
for _, r in apt_piv_r.nlargest(5, "y2019").iterrows():
    axes[1].annotate(r["APT_NAME"],
                     xy=(r["y2019"] / 1e3, r["y2024"] / 1e3),
                     fontsize=7, ha="left",
                     xytext=(3, 3), textcoords="offset points")

plt.tight_layout()
save_figure(fig, "airport_volume_distribution_2019_2024")
plt.show()

### 3.6 Day-of-Week × Month Heatmap (2024)

In [ ]:
hw     = df[df["YEAR"] == 2024].copy()
hw_agg = hw.groupby(["DOW", "MONTH_NUM"])["FLT_TOT_1"].mean().reset_index()
hw_piv = hw_agg.pivot(index="DOW", columns="MONTH_NUM", values="FLT_TOT_1")

DOW_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
MON_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
              "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    hw_piv,
    annot=True, fmt=",.0f",
    cmap="YlOrRd",
    xticklabels=MON_LABELS,
    yticklabels=DOW_LABELS,
    ax=ax,
    linewidths=0.4,
    linecolor="white",
    cbar_kws={"label": "Avg flights / day", "shrink": 0.8},
)
ax.set_title("Average traffic by day-of-week and month (2024)")
ax.set_ylabel("")
ax.set_xlabel("")
plt.tight_layout()
save_figure(fig, "airport_dow_month_heatmap_2024")
plt.show()